In [1]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

In [2]:
// ==== From N02 ====

interface MyMap<K, V> {
    // return the previous value of key, or null if key was not in the map
    V put(K key, V value);
    V remove(K key);

    // null if key is not in the map
    V get(K key);
    boolean containsKey(K key);

    boolean isEmpty();
    int size();
}

class HPoint {
    final int x, y;

    HPoint(int x, int y) { this.x = x; this.y = y; }

    @Override
    public boolean equals(Object o) {
        if (!(o instanceof HPoint)) return false;
        HPoint p = (HPoint) o;
        return x == p.x && y == p.y;
    }

    @Override
    public int hashCode() { return 31 * x + y; }     // or Objects.hash(x, y)
}

# Collisions

## Two strategies

- **Separate chaining**: each cell holds a linked list (**chain**) of pairs.
- **Open addressing**: one pair per cell. On a collision, look for another cell.

## Separate chaining

- `put`, `get`, `remove`: compute `i = hash(k) mod M`, then work on chain `i` only.
- `put`: if `k` is in chain `i`, replace its value; otherwise add a new node.
- Example: `M = 5`, `hash(k) = k`. `put` 11, 7, 12, 22. Then `get(22)`: how many comparisons?

## Load factor

- `α = N / M`: number of **pairs** over number of cells (not non-empty cells).
- Average chain length, over all `M` cells: exactly `α`, for any hash function.
- `get(k)` scans only the chain of `k`: its cost depends on the length of **that** chain.
- Uniform hashing (each key equally likely to land in any cell, independently of the others): the chain of `k` has expected length about `α`.
  - `get` costs `O(1 + α)` on average: `1` to compute `hash(k)` and access the cell, `α` to scan the chain.
  - Not `O(α)`: `α` can be arbitrarily small (large `M`, or many removals), and then the constant term dominates. Every `get` computes `hash(k)` and reads a cell, even if the chain is empty.
- Keep `α` bounded (e.g. `α ≤ 0.75`): `O(1)` on average.
- Worst case: all keys in the same chain. The average chain length is still `α`, but `get` costs `O(N)` in the worst case.

## Rehashing

- `α` above the threshold: new array, twice as large.
- Every pair moves to `hash(k) mod 2M`. Why not copy each cell into the same position?
- One rehash: `O(N)`. `put`: `O(1)` amortized, as in `MyArrayListDouble`.

In [3]:
class ChainedHashMap<K, V> implements MyMap<K, V> {
    private static class Node<K, V> {
        K key;
        V value;
        Node<K, V> next;        // next node in the same chain

        Node(K key, V value, Node<K, V> next) {
            this.key = key;
            this.value = value;
            this.next = next;
        }
    }

    private static final double MAX_LOAD = 0.75;

    private Node<K, V>[] table;     // table[i]: first node of chain i, null if empty
    private int size;

    ChainedHashMap() { this(8); }

    @SuppressWarnings("unchecked")
    ChainedHashMap(int capacity) { table = (Node<K, V>[]) new Node[capacity]; }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    // null keys are not allowed: key.hashCode() throws
    private int index(K key) { return Math.floorMod(key.hashCode(), table.length); }

    // the node of key, or null
    private Node<K, V> find(K key) {
        for (Node<K, V> n = table[index(key)]; n != null; n = n.next)
            if (n.key.equals(key)) return n;
        return null;
    }

    public V get(K key) {
        Node<K, V> n = find(key);
        return n == null ? null : n.value;
    }

    public boolean containsKey(K key) { return find(key) != null; }

    public V put(K key, V value) {
        Node<K, V> n = find(key);
        if (n != null) {                // key already there: replace the value
            V old = n.value;
            n.value = value;
            return old;
        }
        int i = index(key);
        table[i] = new Node<>(key, value, table[i]);    // at the front of chain i
        size++;
        if (size > MAX_LOAD * table.length) rehash();
        return null;
    }

    public V remove(K key) {
        int i = index(key);
        Node<K, V> prev = null;
        for (Node<K, V> n = table[i]; n != null; prev = n, n = n.next) {
            if (n.key.equals(key)) {
                if (prev == null) table[i] = n.next;
                else prev.next = n.next;
                size--;
                return n.value;
            }
        }
        return null;
    }

    @SuppressWarnings("unchecked")
    private void rehash() {
        Node<K, V>[] old = table;
        table = (Node<K, V>[]) new Node[2 * old.length];
        for (Node<K, V> chain : old) {
            Node<K, V> n = chain;
            while (n != null) {
                Node<K, V> next = n.next;
                int i = index(n.key);   // index in the new table
                n.next = table[i];      // move n to the front of chain i
                table[i] = n;
                n = next;
            }
        }
    }

    @Override
    public String toString() {
        StringJoiner sj = new StringJoiner(", ", "{", "}");
        for (Node<K, V> chain : table)
            for (Node<K, V> n = chain; n != null; n = n.next)
                sj.add(n.key + "=" + n.value);
        return sj.toString();
    }

    // one line per cell: its chain, from the front
    public String internals() {
        StringBuilder sb = new StringBuilder();
        for (int i = 0; i < table.length; i++) {
            sb.append(i).append(":");
            for (Node<K, V> n = table[i]; n != null; n = n.next)
                sb.append(" [").append(n.key).append("|").append(n.value).append("]");
            sb.append("\n");
        }
        return sb.toString();
    }
}

In [ ]:
ChainedHashMap<Integer, String> cm = new ChainedHashMap<>(5);
cm.put(11, "A");
cm.put(7, "B");
cm.put(12, "C");
System.out.println(cm.internals());

cm.put(22, "D");                // 4 > 0.75 * 5: rehash
System.out.println(cm.internals());

System.out.println(cm.put(12, "X") + " " + cm.get(12) + " " + cm.remove(7) + " " + cm.get(7) + " " + cm);

- New nodes go at the **front** of the chain. Why not at the back?
- Cost of `put`, `get`, `remove`: on average? Worst case? When does the worst case happen?

In [ ]:
// N lookups, each for a key that is present
System.out.printf("%8s %12s %12s %12s%n", "N", "list ms", "ours ms", "HashMap ms");
for (int n = 5_000; n <= 40_000; n *= 2) {
    final int N = n;
    List<Integer> list = new ArrayList<>();
    MyMap<Integer, Integer> ours = new ChainedHashMap<>();
    Map<Integer, Integer> theirs = new HashMap<>();
    for (int i = 0; i < N; i++) {
        list.add(i);
        ours.put(i, i);
        theirs.put(i, i);
    }
    long l = millis(() -> { for (int i = 0; i < N; i++) list.contains(i); });
    long o = millis(() -> { for (int i = 0; i < N; i++) ours.containsKey(i); });
    long t = millis(() -> { for (int i = 0; i < N; i++) theirs.containsKey(i); });
    System.out.printf("%,8d %12d %12d %12d%n", N, l, o, t);
}

- Keys `0..N-1` and `hash(k) = k`: why is this the best case for our map?

## Open addressing

- All pairs are stored in the array itself: one pair per cell, no lists, no nodes.
- On a collision, the pair goes to another cell, chosen by a fixed rule: the **probe sequence** of the key, `h(k, 0), h(k, 1), h(k, 2), ...`, starting from the **home cell** of `k`, `h(k, 0) = hash(k) mod M`: the cell where `k` goes if there is no collision.
- Every operation on `k` follows the same probe sequence: `get` looks exactly where `put` may have placed `k`.
- Each pair takes a whole cell: `α = N / M < 1` always.

## Linear probing

- `h(k, j) = (hash(k) + j) mod M`: try the next cell, then the one after, wrapping around at the end of the array.
- `put(k, v)`: follow the probe sequence.
  - A cell holding `k`: replace its value.
  - An empty cell: `k` is not in the table. Store the pair there.
- `get(k)`: follow the probe sequence.
  - A cell holding `k`: found.
  - An empty cell: `k` is not in the table, since `put` would have stopped at this cell.
- Example: `M = 5`, `hash(k) = k`. `put` 11, 7, 12, 22. Then:
  - `get(22)`: how many cells?
  - `put(3, "E")`: which cell?

### Clustering

- **Cluster**: a run of consecutive occupied cells.
- A key whose home cell is anywhere in a cluster ends up right after it: the cluster grows by one cell.
- A long cluster is hit more often than a short one, so it grows faster (**primary clustering**). Neighboring clusters merge.
- A search that starts inside a cluster scans it up to the key or to its end.

Expected number of cells visited, keys spread evenly (Knuth):

| `α` | 0.25 | 0.5 | 0.75 | 0.9 |
|---|---|---|---|---|
| `get(k)`, `k` present: `½ (1 + 1/(1-α))` | 1.2 | 1.5 | 2.5 | 5.5 |
| `get(k)`, `k` absent: `½ (1 + 1/(1-α)²)` | 1.4 | 2.5 | 8.5 | 50.5 |

- Fine up to `α = 0.5`, then grows quickly: `α` is usually kept below `0.5`.

In [4]:
class LinearProbingMap<K, V> {
    private static final double MAX_LOAD = 0.5;

    private K[] keys;           // keys[i] == null: cell i is empty
    private V[] values;
    private int size;

    LinearProbingMap() { this(8); }

    @SuppressWarnings("unchecked")
    LinearProbingMap(int capacity) {
        keys = (K[]) new Object[capacity];
        values = (V[]) new Object[capacity];
    }

    public int size() { return size; }

    private int index(K key) { return Math.floorMod(key.hashCode(), keys.length); }

    // the cell of key if it is there, otherwise the empty cell where the search stops
    private int probe(K key) {
        int i = index(key);
        while (keys[i] != null && !keys[i].equals(key))
            i = (i + 1) % keys.length;
        return i;
    }

    public V get(K key) { return values[probe(key)]; }     // null if the cell is empty

    public V put(K key, V value) {
        int i = probe(key);
        V old = values[i];
        if (keys[i] == null) {          // new key
            keys[i] = key;
            size++;
        }
        values[i] = value;
        if (size > MAX_LOAD * keys.length) rehash();
        return old;
    }

    @SuppressWarnings("unchecked")
    private void rehash() {
        K[] oldKeys = keys;
        V[] oldValues = values;
        keys = (K[]) new Object[2 * oldKeys.length];
        values = (V[]) new Object[2 * oldKeys.length];
        size = 0;
        for (int i = 0; i < oldKeys.length; i++)
            if (oldKeys[i] != null) put(oldKeys[i], oldValues[i]);
    }

    public String internals() { return Arrays.toString(keys); }
}

In [ ]:
LinearProbingMap<Integer, String> lp = new LinearProbingMap<>(10);
lp.put(11, "A");
lp.put(7, "B");
lp.put(12, "C");
lp.put(22, "D");
lp.put(3, "E");
System.out.println(lp.internals() + "  " + lp.get(22) + " " + lp.get(3) + " " + lp.get(5));

lp.put(13, "F");                // 6 > 0.5 * 10: rehash
System.out.println(lp.internals());

- `probe` loops forever: when? Why can it not happen here?
- `rehash` calls `put`. Why not copy each key into the same cell?

## Quadratic probing

- `h(k, j) = (hash(k) + j²) mod M`: offsets 1, 4, 9, 16, ... from the home cell.
- Two keys with different home cells in the same cluster follow different probe sequences: no primary clustering.
- Two keys with the **same** home cell still follow the same probe sequence (**secondary clustering**).
- The probe sequence may miss empty cells: `M = 5`, 11, 7, 12 in cells 1, 2, 3. `put(22)` tries cells `2, 3, 1, 1, 3, 2, 3, ...`: cells 0 and 4 are empty, but never tried.
- `M` prime: the first `(M+1)/2` cells of a probe sequence are all different. With `α < 1/2`, at least one of them is empty: `put` always succeeds.

## Double hashing

- A second hash function `hash₂` gives the **step**: `h(k, j) = (hash(k) + j · hash₂(k)) mod M`.
- Keys with the same home cell usually have different steps: no secondary clustering.
- `hash₂(k)` must never be `0`. Why?
- `M` prime, step in `1..M-1`: the probe sequence visits every cell. A common choice: `hash₂(k) = 1 + (hash(k) mod (M - 1))`.
- Price: a second hash, and probes jump around the array (fewer consecutive cells than linear probing).

## Removing with open addressing

- Linear probing, `M = 5`, `hash(k) = k`: 11, 7, 12, 22, 3 in cells 1, 2, 3, 4, 0.
- `remove(12)`: set cell 3 to empty. Then `get(3)`? `get(22)`?
- An empty cell means "the search can stop here". Emptying a cell cuts the probe sequences that pass through it: the keys placed after it can no longer be found.
- **Lazy deletion** ("tombstones"): the cell is marked as *deleted*, not empty.
  - `get`: treats a tombstone as an occupied cell that does not match, and continues past it.
  - `put`: may store a new pair in the first tombstone of the probe sequence, but only after checking that `k` is not further along.
- Many tombstones, few pairs: what goes wrong? How to fix it? Do we need to keep tombstones when we rehash?

## Chaining vs. open addressing

| | separate chaining | open addressing |
|---|---|---|
| storage per pair | one node: key, value, `next` | one cell of the array |
| `α` | can exceed 1 | always `< 1`, kept `≤ 0.5` |
| `remove` | unlink a node | lazy deletion |
| `get`, average | `O(1 + α)` | grows quickly as `α → 1` |
| memory access | follows `next` references, scattered in the heap | consecutive cells (linear probing) |
| worst case | `O(N)` | `O(N)` |
| used by | `java.util.HashMap` | `java.util.IdentityHashMap`, Python `dict`, Rust `HashMap`, Go `map` |

# Hash tables in `java.util`

```
Iterable<E>
 └─ Collection<E>
     ├─ List<E>
     ├─ Queue<E>
     │   └─ Deque<E>
     └─ Set<E>

Map<K, V>
```

| class | implements | |
|---|---|---|
| `HashMap<K, V>` | `Map<K, V>` | separate chaining, `α ≤ 0.75` |
| `HashSet<E>` | `Set<E>` | a `HashMap` with no values |
| `Hashtable<K, V>` | `Map<K, V>` | older version of `HashMap`: do not use |

- `HashMap`: `M` is a power of 2. A chain longer than 8 becomes a balanced search tree (later in the course).
- `Set<E>`: no duplicates, no positions. `add`, `remove`, `contains`.

In [ ]:
Set<String> seen = new HashSet<>();
for (String w : "to be or not to be".split(" "))
    if (!seen.add(w)) System.out.println("repeated: " + w);    // add returns false if w was there
System.out.println(seen + " " + seen.contains("be") + " " + seen.size());

- Cost of iterating over all the keys of a `HashMap`: `O(N)`? `O(N + M)`?
- What does a sorted array do that a hash table cannot?
- Do we need to store the keys at all? Which operations need them?

## Exercises

### 1. Word counts
`Map<String, Integer> wordCounts(String text)`: how many times each word appears.
- `getOrDefault(k, d)`: the value of `k`, or `d` if `k` is not there.
- Then: the most frequent word.

In [ ]:
Map<String, Integer> wordCounts(String text) {
    Map<String, Integer> counts = new HashMap<>();
    for (String w : text.toLowerCase().split("\\W+"))     // split on non-letters
        counts.put(w, counts.getOrDefault(w, 0) + 1);
    return counts;
}

In [ ]:
Map<String, Integer> wc = wordCounts("The cat and the hat. The end, and that is that.");
System.out.println(wc);
check(wc.get("the") == 3 && wc.get("that") == 2 && wc.get("cat") == 1, "counts");
check(wc.get("dog") == null, "absent word");

In [ ]:
// a word with the largest count; null if the map is empty
String mostFrequent(Map<String, Integer> counts) {
    String best = null;
    for (String w : counts.keySet())
        if (best == null || counts.get(w) > counts.get(best)) best = w;
    return best;
}

In [ ]:
check("the".equals(mostFrequent(wc)), "most frequent");
check(mostFrequent(new HashMap<>()) == null, "empty");

### 2. First repeated character
`int firstRepeated(String s)`: the position of the first character that already appeared earlier in `s`; `-1` if there is none.
- With a `Set<Character>`. Cost? Without a set?
- The position of the first character that appears **only once** in `s`: what should the map hold?

In [ ]:
int firstRepeated(String s) {
    Set<Character> seen = new HashSet<>();
    for (int i = 0; i < s.length(); i++)
        if (!seen.add(s.charAt(i))) return i;       // add returns false: seen before
    return -1;
}

In [ ]:
check(firstRepeated("abcbad") == 3, "abcbad");
check(firstRepeated("abc") == -1, "no repetition");
check(firstRepeated("") == -1, "empty");

In [ ]:
// position of the first character that appears only once; -1 if there is none
int firstUnique(String s) {
    Map<Character, Integer> count = new HashMap<>();
    for (char c : s.toCharArray())
        count.put(c, count.getOrDefault(c, 0) + 1);
    for (int i = 0; i < s.length(); i++)
        if (count.get(s.charAt(i)) == 1) return i;
    return -1;
}

In [ ]:
check(firstUnique("abcbad") == 2, "abcbad");
check(firstUnique("aabb") == -1, "no unique character");
check(firstUnique("") == -1, "empty");

### 3. Two sum
`int[] twoSum(int[] a, int target)`: indices `{i, j}`, `i < j`, with `a[i] + a[j] == target`; `null` if there are none.
- Two nested loops: cost?
- Sort, then two indices from the two ends: cost? What is lost?
- One pass with a `Map<Integer, Integer>`, value to index: cost?

In [ ]:
int[] twoSum(int[] a, int target) {
    Map<Integer, Integer> indexOf = new HashMap<>();
    for (int j = 0; j < a.length; j++) {
        Integer i = indexOf.get(target - a[j]);    // the other half, among a[0..j-1]
        if (i != null) return new int[]{i, j};
        indexOf.put(a[j], j);
    }
    return null;
}

In [ ]:
check(Arrays.equals(twoSum(new int[]{3, 9, 4, 7}, 11), new int[]{2, 3}), "4 + 7");
check(Arrays.equals(twoSum(new int[]{5, 5}, 10), new int[]{0, 1}), "same value twice");
check(twoSum(new int[]{5, 1}, 10) == null, "none");

### 4. Anagrams
`Map<String, List<String>> anagrams(List<String> words)`: group the words that are anagrams of each other.
- Which key? Two anagrams must give the same key, two non-anagrams different keys.
- Cost, for `N` words of length at most `L`?

In [ ]:
Map<String, List<String>> anagrams(List<String> words) {
    Map<String, List<String>> groups = new HashMap<>();
    for (String w : words) {
        char[] letters = w.toCharArray();
        Arrays.sort(letters);
        String key = new String(letters);           // same letters, same key
        List<String> group = groups.get(key);
        if (group == null) {
            group = new ArrayList<>();
            groups.put(key, group);
        }
        group.add(w);
    }
    return groups;
}

In [ ]:
Map<String, List<String>> g = anagrams(List.of("listen", "stone", "silent", "notes", "enlist", "cat"));
System.out.println(g.values());
check(g.size() == 3, "three groups");
check(g.values().contains(List.of("listen", "silent", "enlist")), "listen, silent, enlist");

### 5. A bad `hashCode`
`hashCode()` returns `42` for every key.
- Shape of the table? Cost of `N` calls to `add`?
- Predict the timings below, then run them.

In [ ]:
class Key {
    final int id;

    Key(int id) { this.id = id; }

    @Override
    public boolean equals(Object o) { return o instanceof Key && ((Key) o).id == id; }

    @Override
    public int hashCode() { return 42; }
}

System.out.printf("%8s %10s %10s%n", "N", "Key ms", "HPoint ms");
for (int n = 2_000; n <= 16_000; n *= 2) {
    final int N = n;
    long bad = millis(() -> {
        Set<Key> s = new HashSet<>();
        for (int i = 0; i < N; i++) s.add(new Key(i));
    });
    long good = millis(() -> {
        Set<HPoint> s = new HashSet<>();
        for (int i = 0; i < N; i++) s.add(new HPoint(i, 0));
    });
    System.out.printf("%,8d %10d %10d%n", N, bad, good);
}

### 6. Removing from `LinearProbingMap`
Add `V remove(K key)`, with tombstones.
- Where to store the tombstones? A `boolean[] deleted`, or a special key?
- Changes to `probe`, `put`, `rehash`?
- Should `size` count the tombstones? Should the load factor?

In [ ]:
class LazyProbingMap<K, V> {
    private static final Object DELETED = new Object();     // tombstone: never equal to a key
    private static final double MAX_LOAD = 0.5;

    private Object[] keys;      // null: empty, DELETED: tombstone
    private V[] values;
    private int size;           // pairs only
    private int used;           // pairs + tombstones: this is what the load factor counts

    LazyProbingMap() { this(8); }

    @SuppressWarnings("unchecked")
    LazyProbingMap(int capacity) {
        keys = new Object[capacity];
        values = (V[]) new Object[capacity];
    }

    public int size() { return size; }

    private int index(Object key) { return Math.floorMod(key.hashCode(), keys.length); }

    // the cell of key, or -1: tombstones do not stop the search
    private int find(K key) {
        for (int i = index(key); keys[i] != null; i = (i + 1) % keys.length)
            if (keys[i] != DELETED && keys[i].equals(key)) return i;
        return -1;
    }

    public V get(K key) {
        int i = find(key);
        return i < 0 ? null : values[i];
    }

    public V put(K key, V value) {
        int i = find(key);
        if (i >= 0) {                   // key already there: replace the value
            V old = values[i];
            values[i] = value;
            return old;
        }
        i = index(key);
        while (keys[i] != null && keys[i] != DELETED)   // first empty cell or tombstone
            i = (i + 1) % keys.length;
        if (keys[i] == null) used++;    // reusing a tombstone: used does not change
        keys[i] = key;
        values[i] = value;
        size++;
        if (used > MAX_LOAD * keys.length) rehash();
        return null;
    }

    public V remove(K key) {
        int i = find(key);
        if (i < 0) return null;
        V old = values[i];
        keys[i] = DELETED;
        values[i] = null;
        size--;
        return old;
    }

    // drops the tombstones; doubles only if the pairs alone fill a quarter of the table
    @SuppressWarnings("unchecked")
    private void rehash() {
        Object[] oldKeys = keys;
        V[] oldValues = values;
        int capacity = size > MAX_LOAD * oldKeys.length / 2 ? 2 * oldKeys.length : oldKeys.length;
        keys = new Object[capacity];
        values = (V[]) new Object[capacity];
        size = 0;
        used = 0;
        for (int i = 0; i < oldKeys.length; i++)
            if (oldKeys[i] != null && oldKeys[i] != DELETED) put((K) oldKeys[i], oldValues[i]);
    }

    public String internals() {
        StringJoiner sj = new StringJoiner(", ", "[", "]");
        for (Object k : keys) sj.add(k == DELETED ? "DEL" : String.valueOf(k));
        return sj.toString();
    }
}

In [ ]:
LazyProbingMap<Integer, String> lz = new LazyProbingMap<>(10);
lz.put(11, "A");
lz.put(7, "B");
lz.put(12, "C");
lz.put(22, "D");
lz.put(3, "E");
System.out.println(lz.internals());

lz.remove(12);                  // cell 2: tombstone
System.out.println(lz.internals() + "  " + lz.get(22) + " " + lz.get(3) + " " + lz.get(12));

lz.put(32, "F");                // not found, then reuses the tombstone in cell 2
System.out.println(lz.internals());

for (int i = 0; i < 100_000; i++) {
    lz.put(1000 + i, "x");
    lz.remove(1000 + i);
}
check(lz.size() == 5 && lz.get(32).equals("F") && lz.get(1000) == null, "many put and remove");

### 7. By hand
- `M = 7`, `hash(k) = k`: `put` 3, 10, 17, 24, 31, with linear probing and with quadratic probing. Where does 31 go? `α`?
- Linear probing, `M = 7`: keys 21, 15, 38, 12, 76. Draw the table.
  - `put(16)`: `α = 6/7`, rehash into `M = 15`. Draw the new table.

Linear probing: 3, 10, 17, 24, 31 in cells 3, 4, 5, 6, 0. `α = 5/7`.

Quadratic probing: squares mod 7 are `0, 1, 4, 2, 2, 4, 1, 0, ...`, so every key probes cells `3, 4, 0, 5, 5, 0, 4, 3, ...`
- 3, 10, 17, 24 in cells 3, 4, 0, 5.
- 31: never placed. Cells 1, 2, 6 are empty but never tried. `α = 4/7 > 1/2`: no guarantee.

Rehashing, `M = 7`:
```
  0    1    2    3    4    5    6
| 21 | 15 |    | 38 |    | 12 | 76 |
```
`put(16)`: `16 mod 7 = 2`, cell 2. Old cells reinserted in order 0..6 into `M = 15`: 21→6, 15→0, 16→1, 38→8, 12→12, 76→1, taken, →2.
```
  0    1    2    3    4    5    6    7    8    9   10   11   12   13   14
| 15 | 16 | 76 |    |    |    | 21 |    | 38 |    |    |    | 12 |    |    |
```
- 16 inserted after the rehash instead: 76 in cell 1, 16 in cell 2.